In [ ]:
import os
import glob
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

#Device agnostic
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"aktif cihaz: {device}")")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def prepare_data_split(dataset_path, test_monkey):
    all_files = sorted(glob.glob(os.path.join(dataset_path, "*.npz")))
    train_files, test_files = [], []

    for file_path in all_files:
        filename = os.path.basename(file_path).lower()
        if "post" not in filename:
            continue
        if test_monkey.lower() in filename:
            test_files.append(file_path)
        else:
            train_files.append(file_path)

    print(f"   Train Set : {len(train_files)} dosya")
    print(f"   Test Set ({test_monkey.upper()}): {len(test_files)} dosya")
    return train_files, test_files

In [ ]:
class ASSREegDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        npz_data = np.load(self.file_paths[idx])

        spect_matrix = npz_data['spect']

        itpc_matrix = np.concatenate([
            npz_data['delta'],
            npz_data['theta'],
            npz_data['alpha'],
            npz_data['beta'],
            npz_data['gamma']
        ], axis=1)

        label = int(npz_data['label'])

        spect_tensor = torch.tensor(spect_matrix, dtype=torch.float32)
        itpc_tensor = torch.tensor(itpc_matrix, dtype=torch.float32)
        y_tensor = torch.tensor(label, dtype=torch.long)

        return itpc_tensor, spect_tensor, y_tensor




In [ ]:
class ASSR2DCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(ASSR2DCNN, self).__init__()

        self.spect_norm = nn.InstanceNorm2d(7)
        self.spect_features = nn.Sequential(
            nn.Conv2d(7, 16, kernel_size=3, padding=1),
            nn.InstanceNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.InstanceNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d((4, 16))
        )

        self.itpc_norm = nn.InstanceNorm2d(7)
        self.itpc_features = nn.Sequential(
            nn.Conv2d(7, 16, kernel_size=3, padding=1),
            nn.InstanceNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.InstanceNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d((4, 16))
        )

        self.classifier = nn.Sequential(
            nn.Linear(8192, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, itpc, spectrogram):
        x_spect = self.spect_norm(spectrogram)
        x_spect = self.spect_features(x_spect)
        x_spect = torch.flatten(x_spect, 1)

        x_itpc = self.itpc_norm(itpc)
        x_itpc = self.itpc_features(x_itpc)
        x_itpc = torch.flatten(x_itpc, 1)

        combined = torch.cat((x_spect, x_itpc), dim=1)
        out = self.classifier(combined)
        return out




In [ ]:
def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=15):
    for epoch in range(num_epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for itpc, spectrogram, labels in train_loader:
            itpc, spectrogram, labels = itpc.to(device), spectrogram.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(itpc, spectrogram)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100 * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        test_loss, test_correct, test_total = 0.0, 0, 0

        with torch.no_grad():
            for itpc, spectrogram, labels in test_loader:
                itpc, spectrogram, labels = itpc.to(device), spectrogram.to(device), labels.to(device)

                outputs = model(itpc, spectrogram)
                loss = criterion(outputs, labels)

                test_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        test_acc = 100 * test_correct / test_total
        avg_test_loss = test_loss / len(test_loader)

        print(
            f"Epoch {epoch + 1:02d}/{num_epochs:02d} | "
            f"Train Loss: {avg_train_loss:.4f} - Acc: %{train_acc:.2f} | "
            f"Test Loss: {avg_test_loss:.4f} - Acc: %{test_acc:.2f}"
        )

def evaluate_model(model, test_loader, test_monkey_name):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for itpc, spectrogram, labels in test_loader:
            itpc, spectrogram, labels = itpc.to(device), spectrogram.to(device), labels.to(device)
            outputs = model(itpc, spectrogram)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\n[{test_monkey_name.upper()}] Test Doğruluğu: %{acc*100:.2f}")

    class_names = ['Saline', 'Perampanel', 'Diazepam']
    print("\nSınıflandırma Raporu:")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Tahmin Edilen')
    plt.ylabel('Gerçek Sınıf')
    plt.title(f'Confusion Matrix - {test_monkey_name.upper()}')
    plt.show()




In [ ]:
if __name__ == "__main__":
    set_seed(1881)

    DRIVE_PATH = "/content/drive/MyDrive/processed_dataset_spectrogram_itpc_withRef/"
    LOCAL_PATH = "/content/local_dataset/"

    # Since I'm computing on colab I first import the files on the local drive, if haven't already.
    if not os.path.exists(LOCAL_PATH):
        print("kopyalaniyor...")
        shutil.copytree(DRIVE_PATH, LOCAL_PATH)
        print("kopyalandi\n")

    DATASET_PATH = LOCAL_PATH
    TEST_MONKEY = "preston"
    BATCH_SIZE = 8
    NUM_EPOCHS = 15

    train_files, test_files = prepare_data_split(DATASET_PATH, TEST_MONKEY)

    if len(train_files) == 0:
        print("check DRIVE_PATH")
    else:
        train_dataset = ASSREegDataset(train_files)
        test_dataset = ASSREegDataset(test_files)

        g = torch.Generator()
        g.manual_seed(1881)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, generator=g)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        model = ASSR2DCNN(num_classes=3).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        print("Training...")
        train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=NUM_EPOCHS)

        print(f"\n(LOSO - {TEST_MONKEY.upper()})...")
        evaluate_model(model, test_loader, TEST_MONKEY)

In [ ]:
# ==========================================================
# Dynamic Hyperparameter and Model Summary Printer
# ==========================================================

print("=" * 60)
print("             MODEL VE EĞİTİM HİPERPARAMETRELERİ            ")
print("=" * 60)

test_monkey_val = globals().get('TEST_MONKEY', 'Bilinmiyor')
num_epochs_val = globals().get('NUM_EPOCHS', 'Bilinmiyor')
dataset_path_val = globals().get('DATASET_PATH', 'Bilinmiyor')

if 'train_loader' in globals() and hasattr(train_loader, 'batch_size'):
    batch_size_val = train_loader.batch_size
else:
    batch_size_val = globals().get('BATCH_SIZE', 'Bilinmiyor')

if 'optimizer' in globals():
    lr_val = optimizer.param_groups[0]['lr']
    opt_name = optimizer.__class__.__name__
else:
    lr_val = "Bilinmiyor"
    opt_name = "Bilinmiyor"

loss_fn_name = criterion.__class__.__name__ if 'criterion' in globals() else "Bilinmiyor"

conv_channels = []
kernel_sizes = set()
pool_sizes = set()
adaptive_pool_sizes = set()
norm_type = "Bilinmiyor"
fc_hidden_dims = []
dropout_rates = []
num_classes_val = "Bilinmiyor"
in_channels_spect = "Bilinmiyor"
combined_feature_dim = "Bilinmiyor"

if 'model' in globals():

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    if hasattr(model, 'spect_norm'):
        norm_type = model.spect_norm.__class__.__name__
        in_channels_spect = getattr(model.spect_norm, 'num_features', 'Bilinmiyor')

    if hasattr(model, 'spect_features'):
        for layer in model.spect_features:
            if isinstance(layer, torch.nn.Conv2d):
                conv_channels.append(layer.out_channels)
                kernel_sizes.add(f"{layer.kernel_size[0]}x{layer.kernel_size[1]} (pad={layer.padding[0]})")
            elif isinstance(layer, torch.nn.MaxPool2d):
                k_size = layer.kernel_size if isinstance(layer.kernel_size, tuple) else (layer.kernel_size, layer.kernel_size)
                pool_sizes.add(f"{k_size[0]}x{k_size[1]}")
            elif isinstance(layer, torch.nn.AdaptiveAvgPool2d):
                adaptive_pool_sizes.add(str(layer.output_size))

    if hasattr(model, 'classifier'):
        linear_layers = [l for l in model.classifier if isinstance(l, torch.nn.Linear)]
        if linear_layers:
            combined_feature_dim = linear_layers[0].in_features  # CNN çıktılarının birleştiği boyut
            fc_hidden_dims = [l.out_features for l in linear_layers[:-1]]  # Gizli katman boyutları
            num_classes_val = linear_layers[-1].out_features  # Çıktı / Sınıf sayısı

        dropouts = [l.p for l in model.classifier if isinstance(l, torch.nn.Dropout)]
        dropout_rates = dropouts
else:
    total_params = "Model bulunamadı"
    trainable_params = "Model bulunamadı"

# Metin Biçimlendirmeleri
kernel_str = ", ".join(kernel_sizes) if kernel_sizes else "Bilinmiyor"
pool_str = ", ".join(pool_sizes) if pool_sizes else "Bilinmiyor"
adaptive_pool_str = ", ".join(adaptive_pool_sizes) if adaptive_pool_sizes else "Bilinmiyor"
fc_dim_str = ", ".join(map(str, fc_hidden_dims)) if fc_hidden_dims else "Yok (Doğrudan Çıktı)"
dropout_str = ", ".join([f"{p} (%{int(p*100)})" for p in dropout_rates]) if dropout_rates else "Kullanılmadı"
model_name = model.__class__.__name__ if 'model' in globals() else "ASSR2DCNN"

# ==========================================================
# Output Table
# ==========================================================
print(f"📌 [Genel ve Veri Ayarları]")
print(f"   • Test Konusu / Maymun (LOSO)     : {str(test_monkey_val).upper()}")
print(f"   • Veri Yolu (Dataset Path)        : {dataset_path_val}")
print(f"   • Sınıf Sayısı (Num Classes)       : {num_classes_val}")

print(f"\n📌 [Eğitim Hiperparametreleri]")
print(f"   • Batch Size (Yığın Boyutu)       : {batch_size_val}")
print(f"   • Num Epochs (Epok Sayısı)        : {num_epochs_val}")
print(f"   • Learning Rate (Öğrenme Oranı)   : {lr_val}")
print(f"   • Optimizer (Optimizer Türü)      : {opt_name}")
print(f"   • Loss Function (Kayıp Fonksiyonu): {loss_fn_name}")

print(f"\n📌 [Model Mimarisi Hiperparametreleri ({model_name})]")
print(f"   • Girdi Kanal Sayıları (SPECT/ITPC): {in_channels_spect} / {in_channels_spect}")
print(f"   • Evrişim Katman Filtreleri (CNN)  : {conv_channels if conv_channels else 'Bilinmiyor'}")
print(f"   • Çekirdek Boyutu (Kernel Size)    : {kernel_str}")
print(f"   • Havuzlama Boyutu (MaxPool)       : {pool_str}")
print(f"   • Uyarlanabilir Havuzlama          : {adaptive_pool_str}")
print(f"   • Normalizasyon Türü               : {norm_type}")
print(f"   • Birleştirilmiş Özellik Vektörü   : {combined_feature_dim}")
print(f"   • Tam Bağlantılı Gizli Katman (FC) : {fc_dim_str}")
print(f"   • Dropout Oranı                    : {dropout_str}")

print(f"\n📌 [Model İstatistikleri]")
print(f"   • Toplam Parametre Sayısı          : {total_params:,}" if isinstance(total_params, int) else f"   • Toplam Parametre Sayısı          : {total_params}")
print(f"   • Eğitilebilir Parametre Sayısı    : {trainable_params:,}" if isinstance(trainable_params, int) else f"   • Eğitilebilir Parametre Sayısı    : {trainable_params}")
print("=" * 60)

In [ ]:
;

In [ ]:
;